# Tree Structure and Traversal

```{contents}
:local:
:depth: 2
```

## Tree Vocabulary

```{index} tree; root; leaf; subtree
```


A **tree** is a collection of nodes connected by parent-child relationships. The **root** is the starting node. A **leaf** has no children. A **subtree** is a node together with everything below it.

Many real systems have this shape: folders contain folders, supervisors manage teams, expressions contain subexpressions, and web pages link to lower-level navigation sections.


In [ ]:
using System;

var root = new TreeNode("Courses");
root.Left = new TreeNode("Programming");
root.Right = new TreeNode("Databases");
root.Left.Left = new TreeNode("C#");
root.Left.Right = new TreeNode("Python");

Console.WriteLine(root.Value);
Console.WriteLine(root.Left.Value);
Console.WriteLine(root.Left.Left.Value);

class TreeNode
{
    public string Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(string value)
    {
        Value = value;
    }
}


This first representation uses `Left` and `Right` references, so it is a **binary tree**. The values could represent labels, numbers, records, or any other data.


A tree models a hierarchy with one route from the root to each node. In a nonempty tree, every node except the root has exactly one parent. There are no cycles, and all nodes belong to the same connected structure. A structure with shared children or a link back to an ancestor needs graph algorithms instead of these tree algorithms.

A company's departments form a general tree: a manager can have more than two direct reports. `Left` and `Right` deliberately restrict the example to two children. The names indicate positions, not a numerical ordering rule. A binary tree can contain unsorted labels or numbers.

For the course catalog example, draw each reference as an edge:

```text
        Courses
       /       \
Programming   Databases
   /    \
 C#    Python
```

`Courses` has depth zero, `Programming` has depth one, and `C#` has depth two. There are five nodes and four edges. Every non-root node contributes its one incoming edge, so a nonempty tree with n nodes has n - 1 edges. An empty tree has no root and zero edges.

Before running the next example, identify all leaves. A node with only one child is an internal node, not a leaf. Two labels can be equal and still belong to different nodes; a node is an object, while its label is stored data.

## Recursive Structure

```{index} tree; recursive structure
```


Trees are recursive because each child is the root of a smaller tree. That means many tree algorithms follow the same pattern:

1. Stop if the current node is `null`.
2. Do some work at the current node.
3. Recur into one or more child subtrees.

This is not recursion for its own sake. It mirrors the structure of the data.


In [ ]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

Console.WriteLine($"Node count: {Count(root)}");
Console.WriteLine($"Leaf count: {CountLeaves(root)}");

int Count(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    return 1 + Count(node.Left) + Count(node.Right);
}

int CountLeaves(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    if (node.Left is null && node.Right is null)
    {
        return 1;
    }

    return CountLeaves(node.Left) + CountLeaves(node.Right);
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


`Count` treats the current node as one item plus the size of each child subtree. `CountLeaves` changes the base case: a node with no children contributes one leaf.


The recursion needs two guarantees. First, a null reference represents an empty subtree and contributes zero. Second, each recursive call receives a smaller subtree. In a finite tree with no cycles, repeated calls eventually reach null. A cycle would break this termination argument.

For the subtree rooted at 3, `Count(3)` waits for `Count(1)` and `Count(6)`. Both return one, so the call returns `1 + 1 + 1 = 3`. The call at 8 combines that result with the one node rooted at 10 and returns five. Each return summarizes a subtree; it does not count the same node again.

| Call | Left result | Right result | Returned count |
| --- | ---: | ---: | ---: |
| Count(1) | 0 | 0 | 1 |
| Count(6) | 0 | 0 | 1 |
| Count(3) | 1 | 1 | 3 |
| Count(10) | 0 | 0 | 1 |
| Count(8) | 3 | 1 | 5 |

`CountLeaves` has a different meaning: an internal node contributes no leaf of its own, while a node with two null children contributes one. That distinction explains why adding one unconditionally would count nodes instead of leaves.

## Traversal Orders

```{index} preorder traversal; inorder traversal; postorder traversal
```


A **traversal** visits every node in a tree. For binary trees, the common depth-first traversal orders differ only in when the current node is processed:

- **Preorder**: node, left, right
- **Inorder**: left, node, right
- **Postorder**: left, right, node


In [ ]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

Console.Write("Preorder:  ");
Preorder(root);
Console.WriteLine();

Console.Write("Inorder:   ");
Inorder(root);
Console.WriteLine();

Console.Write("Postorder: ");
Postorder(root);
Console.WriteLine();

void Preorder(TreeNode? node)
{
    if (node is null) return;
    Console.Write(node.Value + " ");
    Preorder(node.Left);
    Preorder(node.Right);
}

void Inorder(TreeNode? node)
{
    if (node is null) return;
    Inorder(node.Left);
    Console.Write(node.Value + " ");
    Inorder(node.Right);
}

void Postorder(TreeNode? node)
{
    if (node is null) return;
    Postorder(node.Left);
    Postorder(node.Right);
    Console.Write(node.Value + " ");
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


The tree shape is the same in all three traversals. Only the timing of `Console.Write` changes.


Trace the same five-node tree with the root at 8. In preorder, you process 8 before entering the call for 3. In inorder, the call at 8 stays suspended until the entire left subtree returns. In postorder, both subtrees finish before 8 is processed.

| Order | Sequence | Use in a hierarchy |
| --- | --- | --- |
| Preorder | 8, 3, 1, 6, 10 | Print a department heading before its teams. |
| Inorder | 1, 3, 6, 8, 10 | List keys in order when the tree is a BST. |
| Postorder | 1, 6, 3, 10, 8 | Total child budgets before computing a department total. |

Inorder traversal is sorted here because this particular tree obeys the binary search tree rule. Inorder traversal of an arbitrary binary tree is not necessarily sorted.

Every traversal visits each node exactly once. If processing a node costs constant time, total work is Θ(n). Recursive calls retain their local state while waiting for descendants, so the largest number of active calls is proportional to height: O(h + 1) extra space for a nonempty tree. Returning a list of all values also uses Θ(n) space for that result. The output list and the call stack are different costs.

**Predict:** move 6 from the right child of 3 to the left child of 10. Which traversal sequences change? The node count stays the same, but the order depends on the references, not only the stored values.

## Height and Shape

```{index} tree height; tree depth
```


A node's **depth** is its distance from the root. A tree's **height** is the longest downward path from the root to a leaf. Height matters because many tree operations follow one path from the root toward a leaf.


In [ ]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);
root.Left.Right.Left = new TreeNode(4);

Console.WriteLine($"Height: {Height(root)}");

int Height(TreeNode? node)
{
    if (node is null)
    {
        return -1;
    }

    return 1 + Math.Max(Height(node.Left), Height(node.Right));
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


This version defines an empty tree as height `-1`, so a single-node tree has height `0`. Some texts define height differently; always check the convention being used.


Height counts edges in this chapter. Null has height -1, a leaf has height zero, and every internal node adds one to the larger child height. For the six-node example, the call at 6 returns one, the call at 3 returns two, and the root returns three. Depth belongs to a specific node; height summarizes a subtree.

Height computation visits both children, so it costs Θ(n), even when the tree is short. A search may follow only one path; computing the height of an uncached tree examines every node. In a chain of n nodes, height is n - 1 and recursion can require n active calls. Large adversarial chains can exhaust the call stack. Production trees often control height or use iterative algorithms.

**Check the boundaries:** an empty tree has zero nodes, zero leaves, and height -1. A singleton has one node, one leaf, and height zero. A chain with several nodes still has only one leaf. These cases distinguish correct base cases from methods that merely work on a symmetric example.

## Exercise

```{index} tree traversal; exercise
```


In [ ]:
// Exercise: Count leaves
// Complete CountLeaves so the program prints 3 for this tree.

using System;

var root = new TreeNode(5);
root.Left = new TreeNode(2);
root.Right = new TreeNode(9);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(3);

Console.WriteLine(CountLeaves(root));

int CountLeaves(TreeNode? node)
{
    // Your code starts here.



    // Your code ends here.
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


In [1]:
// Solution
using System;

var root = new TreeNode(5);
root.Left = new TreeNode(2);
root.Right = new TreeNode(9);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(3);

Console.WriteLine(CountLeaves(root));

int CountLeaves(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    if (node.Left is null && node.Right is null)
    {
        return 1;
    }

    return CountLeaves(node.Left) + CountLeaves(node.Right);
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


3


## Summary

```{index} Summary
```


Tree algorithms usually follow the data: handle the current node, then recur into child subtrees. Traversal order controls when the current node is processed.

```{rubric} Footnotes
```

[^1]: This chapter uses binary trees so examples stay focused. General trees can have any number of children.
